# mRAG — Pipeline test report

Two tiers, matching `tests/unit/` and `tests/integration/`:
- **Tier 1 — Unit** (model-free, fast): deterministic NLU/retrieval logic.
- **Tier 2 — Integration** (end-to-end, loads SigLIP/Qwen/DeBERTa + Qdrant): real query → result scenarios that previously regressed.

Each tier is shown as observable tables; the `pytest` gate output is saved to `tests/test_report.txt`.

## Tier 1 — Unit gate (`python -m pytest`)

In [1]:
import os, sys, subprocess

ROOT = os.getcwd()
while not os.path.isdir(os.path.join(ROOT, 'src')) and ROOT != os.path.dirname(ROOT):
    ROOT = os.path.dirname(ROOT)
sys.path.insert(0, ROOT)

res = subprocess.run([sys.executable, '-m', 'pytest', '-q'], cwd=ROOT, capture_output=True, text=True)
report = res.stdout + ('\n' + res.stderr if res.stderr else '')
with open(os.path.join(ROOT, 'tests', 'test_report.txt'), 'w', encoding='utf-8') as f:
    f.write(report)
print('exit code:', res.returncode, '(0 = all passed)')
print(report.strip().splitlines()[-1])

exit code: 0 (0 = all passed)
sys:1: DeprecationWarning: builtin type swigvarlink has no __module__ attribute


### Unit behavior showcase (model-free)

In [2]:
import pandas as pd
from src.backend.core.config import settings
from src.backend.services.catalog import FashionCatalog
from src.backend.services.rag_service import FashionRAGService
from src.scripts.graph.outfit_slots import get_slot

cat = FashionCatalog(settings.meta_file, settings.graph_file, settings.image_dir)
class _StubEmbedder: siglip = None
rag = FashionRAGService(_StubEmbedder(), object(), object(), cat, limit=5)
print('catalog loaded:', len(cat.valid_product_types), 'product types')

D:\miniconda\envs\mRAG\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


catalog loaded: 113 product types


**product_type resolution** (exact → synonym → corpus → SigLIP)

In [3]:
cases = [('jacket', 'Jacket'), ('trouser', 'Trousers'), ('pants', 'Trousers'),
         ('parka', 'Jacket'), ('anorak', 'Jacket'), ('xyzblahnonsense', '')]
rows = []
for term, exp in cases:
    val, src = rag._resolve_product_type(term)
    rows.append({'input': term, 'resolved': val, 'source': src, 'expected': exp, 'ok': '✓' if val == exp else '✗'})
pd.DataFrame(rows)

,input,resolved,source,expected,ok
0,jacket,Jacket,exact,Jacket,✓
1,trouser,Trousers,exact,Trousers,✓
2,pants,Trousers,exact,Trousers,✓
3,parka,Jacket,corpus,Jacket,✓
4,anorak,Jacket,corpus,Jacket,✓
5,xyzblahnonsense,,,,✓


**slot inference** (pairing target must differ from the anchor slot)

In [4]:
qs = [('do you have a shoe to match', '', 'shoe'),
      ('some trousers please', '', 'bottom'),
      ('the jacket one looks good do you have shoe to match', 'outerwear', 'shoe')]
rows = []
for q, excl, exp in qs:
    got = cat.infer_slot_from_text(q, exclude_slot=excl)
    rows.append({'query': q[:46], 'exclude_slot': excl or '-', 'slot': got, 'expected': exp, 'ok': '✓' if got == exp else '✗'})
pd.DataFrame(rows)

,query,exclude_slot,slot,expected,ok
0,do you have a shoe to match,-,shoe,shoe,✓
1,some trousers please,-,bottom,bottom,✓
2,the jacket one looks good do you have shoe to,outerwear,shoe,shoe,✓


**hard vs soft filters** (only product_type + colour are hard gates)

In [5]:
full = {'product_type': 'Jacket', 'colour_group': 'Black', 'fit': 'Slim/Tailored',
        'occasion': 'Casual/Everyday', 'seasonality': 'Spring/Summer'}
pd.DataFrame([{'stage': 'extracted (all)', 'filters': full}, {'stage': 'hard filters used', 'filters': rag._hard_filters(full)}])

,stage,filters
0,extracted (all),"{'product_type': 'Jacket', 'colour_group': 'Bl..."
1,hard filters used,"{'product_type': 'Jacket', 'colour_group': 'Bl..."


**compat pairing** — shoe complements for a cold jacket (not in co-buy graph)

In [6]:
from src.backend.services.compat_index import CompatPairingIndex
ci = CompatPairingIndex(settings.compat_dir)
A = '0549330005'
print('anchor:', cat.get_meta(A).get('product_type_name'), cat.get_meta(A).get('colour_group_name'), '| in co-buy graph:', A in cat.graph_adj)
rows = [{'article_id': aid, 'product_type': cat.get_meta(aid).get('product_type_name'), 'colour': cat.get_meta(aid).get('colour_group_name')}
        for aid in ci.complement_ids(A, 8, target_slot='shoe')]
pd.DataFrame(rows)

anchor: Jacket Dark Blue | in co-buy graph: False


,article_id,product_type,colour
0,0535176003,Boots,Black
1,0783624001,Boots,Black
2,0631870009,Sneakers,Beige
3,0522044001,Boots,Black
4,0707381001,Boots,Black
5,0890386003,Heeled sandals,Beige
6,0580469005,Pumps,Black
7,0859191001,Heels,Black


## Tier 2 — Integration scenarios (end-to-end)

Loads the real models + Qdrant and runs `_prepare_chat` on the scenarios that previously regressed.
Slow (~2 min) and needs a free GPU + built artifacts + an idle Qdrant DB; the cell skips cleanly otherwise.

In [7]:
import time
os.environ.setdefault('PYTORCH_JIT', '0')
rag_full = None
try:
    from src.backend.retrieval.qdrant import QdrantStore
    from src.backend.retrieval.encoders import QueryEncoder, SigLIPEncoder
    from src.backend.retrieval.embeddings import SparseTfidfEncoder
    from src.backend.retrieval.llm import QwenMultimodalService
    from src.backend.services.session_manager import SessionStore
    sparse = SparseTfidfEncoder.load(settings.sparse_model_path) if os.path.exists(settings.sparse_model_path) else None
    store = QdrantStore(settings.db_path, settings.collection_name)
    embedder = QueryEncoder(siglip=SigLIPEncoder(), sparse=sparse)
    rag_full = FashionRAGService(embedder, store, QwenMultimodalService(), cat, limit=5, personalization=None, sessions=SessionStore(3600, 100))
    print('pipeline ready')
except Exception as exc:
    print('integration skipped:', exc)

D:\mRAG_repo\src\backend\retrieval\qdrant.py:13: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Collection <fashion_products> contains 70197 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  self.client = QdrantClient(path=db_path)



Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]


Loading weights:   1%|          | 4/408 [00:00<00:28, 14.03it/s]


Loading weights:  42%|████▏     | 170/408 [00:00<00:00, 519.85it/s]


Loading weights:  61%|██████    | 249/408 [00:00<00:00, 494.69it/s]

...


Loading weights: 100%|██████████| 338/338 [00:04<00:00, 77.05it/s]


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]


Loading weights:   2%|▏         | 5/202 [00:00<00:39,  5.02it/s]


Loading weights: 100%|██████████| 202/202 [00:01<00:00, 200.10it/s]

pipeline ready


In [8]:
def _run(query, anchor='', session='nb'):
    items, _p, log, has_ctx, direct = rag_full._prepare_chat(query, None, session, 'r', time.perf_counter(), confirmed_intent=None, customer_id=None, selected_anchor_id=anchor)
    return items, log, direct

def _row(name, items, log, direct, ok):
    pairs = list(zip([it['product_type'] for it in items], [it['colour_group'] for it in items]))[:3]
    intent = log.get('intent_hint') or (direct or {}).get('type', '')
    return {'scenario': name, 'intent': intent, 'pt_filter': log.get('must_filters', {}).get('product_type', ''),
            'results': ', '.join(t + '/' + c for t, c in pairs) or '(none)', 'ok': '✓' if ok else '✗'}

scen = []
if rag_full is not None:
    sid, st = rag_full.sessions.get_or_create('nb-variant')
    rag_full.sessions.touch_anchor(st, '0569973006', ['0569973006'])

    it, lg, dr = _run('find me trousers')
    scen.append(_row('find trousers', it, lg, dr, lg['must_filters'].get('product_type') == 'Trousers'))

    it, lg, dr = _run('I want to find some dark blue parkas with a hood and a zip for outdoor adventures')
    scen.append(_row('parka → Jacket (OOV)', it, lg, dr, lg['must_filters'].get('product_type') == 'Jacket' and '0549330005' in [x['article_id'] for x in it]))

    it, lg, dr = _run('do you have shoe to match with that?', anchor='0549330005')
    scen.append(_row('shoe pairing (cold)', it, lg, dr, bool(it) and all(get_slot(x['product_type']) == 'shoe' for x in it)))

    it, lg, dr = _run('the jacket one looks good, do you have shoe to match with that?', anchor='0549330005')
    scen.append(_row('anchor-type collision', it, lg, dr, bool(it) and all(get_slot(x['product_type']) == 'shoe' for x in it)))

    it, lg, dr = _run('hello there')
    scen.append(_row('chit-chat', it, lg, dr, (not it) and ((dr or {}).get('type') == 'chit_chat' or lg.get('intent_hint') == 'chit_chat')))

    it, lg, dr = _run('Do you have black color?', session='nb-variant')
    scen.append(_row('color variant (cross-turn)', it, lg, dr, bool(it) and all(x['product_type'] == 'Hoodie' for x in it) and any('Black' in x['colour_group'] for x in it)))

pd.DataFrame(scen) if scen else 'integration skipped (no pipeline)'

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


,scenario,intent,pt_filter,results,ok
0,find trousers,similar_items,Trousers,"Trousers/Dark Blue, Trousers/Black, Trousers/D...",✓
1,parka → Jacket (OOV),similar_items,Jacket,"Jacket/Dark Blue, Jacket/Dark Blue, Jacket/Dar...",✓
2,shoe pairing (cold),graph_pairing,,"Boots/Black, Boots/Black, Boots/Black",✓
3,anchor-type collision,graph_pairing,,"Boots/Black, Boots/Black, Sneakers/Beige",✓
4,chit-chat,chit_chat,,(none),✓
5,color variant (cross-turn),color_variant,,"Hoodie/Black, Hoodie/Black, Hoodie/Black",✓
